# Build the Vision Transformer

In [47]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from torchvision.datasets import CIFAR10
from torchvision import transforms

## 1. Load data

In [17]:
import os

# should point to experiments/vit-cifar10
if os.getcwd() == '/bigdata/md5_t1fada40cc125fa5312a398ad7cb/experiments/vit-cifar10/notebooks':
    os.chdir('..')

assert os.getcwd() == '/bigdata/md5_t1fada40cc125fa5312a398ad7cb/experiments/vit-cifar10'

In [18]:
import os
os.getcwd()

'/bigdata/md5_t1fada40cc125fa5312a398ad7cb/experiments/vit-cifar10'

In [27]:
IMAGE_SIZE = 32
BATCH_SIZE = 64
PATCH_SIZE = 8
IN_CHANNELS = 3
EMB_SIZE = 384

NUM_PATCHES = (IMAGE_SIZE // PATCH_SIZE) ** 2

In [20]:
dataset = CIFAR10(
    root='../../data', 
    train=True, 
    download=False,
    transform=transforms.ToTensor()
)

In [21]:
dataset[0][0].shape # shape of the first image

torch.Size([3, 32, 32])

### Batches of images

In [22]:
dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [23]:
images, labels = next(iter(dataloader))
print(images.shape)
print(labels.shape)

torch.Size([64, 3, 32, 32])
torch.Size([64])


## 2. Get embeddings

In [24]:
from models.embeddings import PatchEmbedding, TokenPositionEmbedding

In [25]:
patch_embedding = PatchEmbedding(
    image_size=IMAGE_SIZE,
    patch_size=PATCH_SIZE,
    in_channels=IN_CHANNELS,
    emb_size=EMB_SIZE
)

In [26]:
x = patch_embedding(images)
x.shape # B x P x E

torch.Size([64, 16, 384])

In [28]:
token_position_embedding = TokenPositionEmbedding(
    num_patches=NUM_PATCHES,
    emb_size=EMB_SIZE
)

In [30]:
x = token_position_embedding(x)
x.shape # B x (P + 1) x E

torch.Size([64, 17, 384])

## 3. Transformer block :)

In [42]:
from models.transformer import Block

In [31]:
NUM_TRANSFORMER_BLOCKS = 10
NUM_ATTENTION_HEADS = 16
DROPOUT = 0.1

In [41]:
b = [1, 2, 3]

print(b)
print(*b)

[1, 2, 3]
1 2 3


In [49]:
blocks = nn.Sequential(
    *[
        Block(
            num_heads=NUM_ATTENTION_HEADS,
            emb_size=EMB_SIZE,
            dropout=DROPOUT
        ) for _ in range(NUM_TRANSFORMER_BLOCKS)
    ])

In [50]:
len(blocks)

10

In [51]:
x = blocks(x)
x.shape # B x (P + 1) x E

torch.Size([64, 17, 384])

## 4. Classification head :)

In [52]:
N_CLASSES = 10
classifier = nn.Linear(EMB_SIZE, N_CLASSES)

In [54]:
cls_token = x[:, 0, :] # extract only the embeddings for the class token
cls_token.shape

torch.Size([64, 384])

In [57]:
logits = classifier(cls_token)
logits.shape

torch.Size([64, 10])

In [61]:
class VisionTransformer(nn.Module):
    def __init__(
        self,
        image_size,
        patch_size,
        in_channels,
        emb_size,
        num_attention_heads,
        dropout,
        num_transformer_blocks,
        n_classes
    ):
        super().__init__()

        assert image_size % patch_size == 0
        assert emb_size % num_attention_heads == 0

        # 1. Patch embedding
        self.patch_embedding = PatchEmbedding(
            image_size=image_size,
            patch_size=patch_size,
            in_channels=in_channels,
            emb_size=emb_size
        )

        # 2. CLS token + positional embeddings
        num_patches = (image_size // patch_size) ** 2

        self.token_position_embedding = TokenPositionEmbedding(
            num_patches=num_patches,
            emb_size=emb_size
        )

        # 3. Transformer encoder blocks
        self.blocks = nn.Sequential(
            *[
                Block(
                    num_heads=num_attention_heads,
                    emb_size=emb_size,
                    dropout=dropout
                )
                for _ in range(num_transformer_blocks)
            ]
        )

        # 4. Final LayerNorm
        self.norm = nn.LayerNorm(emb_size)

        # 5. Classification head
        self.classifier = nn.Linear(emb_size, n_classes)

    def forward(self, x):
        # x: (B, C, H, W)

        x = self.patch_embedding(x)
        # (B, P, E)

        x = self.token_position_embedding(x)
        # (B, P + 1, E)

        x = self.blocks(x)
        # (B, P + 1, E)

        x = self.norm(x)

        # Extract CLS token
        cls_embedding = x[:, 0, :]
        # (B, E)

        logits = self.classifier(cls_embedding)
        # (B, n_classes)

        return logits

In [76]:
model = VisionTransformer(
    image_size=32,
    patch_size=4,
    in_channels=3,
    emb_size=384,
    num_attention_heads=6,
    dropout=0.1,
    num_transformer_blocks=8,
    n_classes=10
)

In [77]:
logits = model(images)
logits.shape

torch.Size([64, 10])

## 5. Next step... training :D

In [78]:
labels

tensor([6, 9, 9, 4, 1, 1, 2, 7, 8, 3, 4, 7, 7, 2, 9, 9, 9, 3, 2, 6, 4, 3, 6, 6,
        2, 6, 3, 5, 4, 0, 0, 9, 1, 3, 4, 0, 3, 7, 3, 3, 5, 2, 2, 7, 1, 1, 1, 2,
        2, 0, 9, 5, 7, 9, 2, 2, 5, 2, 4, 3, 1, 1, 8, 2])

In [79]:
loss = nn.CrossEntropyLoss()(logits, labels)
loss.item()

2.4494566917419434

In [80]:
loss.backward() # First backprop step